# StopTheCap - E1 encoder training (Colab, single T4)

Fine-tunes **distilbert-base-cased** on the Ghostbuster train split, then evaluates across all four
splits with the frozen threshold. Estimated **35-60 minutes** on one T4, well inside a 4 h 40 m
session.

**What it does not do**
- Does not change the split manifest, digest, class weights or threshold policy.
- Does not touch HC3/RAID labels until the final evaluation pass.
- Does not publish anything.

The checkpoint and run log are written to **Drive** every 200 steps, so a disconnect costs at most
200 steps rather than the whole run.

## 1. Configure

In [ ]:
# Private ZIP in your Drive (same archive the measurement notebook used).
ZIP_PATH = "/content/drive/MyDrive/StopTheCap.zip"

REPO_DIR = "/content/StopTheCap"
RUN_DIR  = "/content/drive/MyDrive/StopTheCap_runs/E1"

print("zip     :", ZIP_PATH)
print("repo    :", REPO_DIR)
print("run dir :", RUN_DIR, "(on Drive - survives disconnect)")

## 2. Mount Drive and load the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Changing the runtime type restarts the kernel, which wipes the config cell. Restore sane
# defaults so this cell works either way instead of dying with NameError.
if "LOAD_METHOD" not in globals():
    LOAD_METHOD = "zip"
    ZIP_PATH = "/content/drive/MyDrive/StopTheCap.zip"
    REPO_DIR = "/content/StopTheCap"
    print("NOTE: the config cell had not run in this session; using default paths.")
    print("      Re-run cell 2 if your ZIP lives somewhere else.")

import os, shutil, sys, zipfile
from pathlib import Path

# The probe needs only these. datasets/ is deliberately optional: measurement does not use data.
REQUIRED = ["ml/__init__.py", "ml/cloud/__init__.py", "ml/cloud/probe.py"]
EXTRACT_TMP = "/content/_stc_extract"   # overridden in tests

def _fail(msg):
    raise RuntimeError(msg)

def load_from_zip(zip_path, dest):
    z = Path(zip_path)
    if not z.is_file():
        _fail(
            "ZIP not found: " + str(z) + "\n"
            "Upload your project ZIP to Google Drive, then set ZIP_PATH in the previous cell.\n"
            "Expected form: /content/drive/MyDrive/<name>.zip"
        )
    dest = Path(dest)
    tmp = Path(EXTRACT_TMP)
    for p in (dest, tmp):
        if p.exists():
            shutil.rmtree(p)
    tmp.mkdir(parents=True)

    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print("zip entries:", len(names))
        for n in names[:12]:
            print("   ", n)
        if not names:
            _fail("The ZIP is empty.")
        zf.extractall(tmp)

    entries = [p for p in tmp.iterdir() if p.name != "__MACOSX"]
    # Flatten a single top-level folder so we never end up with StopTheCap/StopTheCap.
    root = entries[0] if len(entries) == 1 and entries[0].is_dir() else tmp
    shutil.move(str(root), str(dest))
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    return dest

def load_from_git(url, dest):
    import subprocess
    if not url:
        _fail("REPO_URL is empty, so git mode cannot run. Use LOAD_METHOD = 'zip'.")
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)
    return dest

if LOAD_METHOD == "zip":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR)
elif LOAD_METHOD == "git":
    REPO = load_from_git(REPO_URL, REPO_DIR)
elif LOAD_METHOD == "auto":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR) if Path(ZIP_PATH).is_file() else load_from_git(REPO_URL, REPO_DIR)
else:
    _fail("Unknown LOAD_METHOD " + repr(LOAD_METHOD) + ". Use 'zip', 'git' or 'auto'.")

REPO = Path(REPO).resolve()
missing = [f for f in REQUIRED if not (REPO / f).is_file()]
if missing:
    _fail(
        "Project structure looks wrong; missing: " + ", ".join(missing) + "\n"
        "The ZIP must contain the repository root - the folder that holds ml/, datasets/, notebooks/."
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print()
print("project root :", REPO)
print("top level    :", sorted(p.name for p in REPO.iterdir())[:14])
if not (REPO / "datasets/manifests/COMBINED.yaml").is_file():
    print("NOTE: datasets/manifests/COMBINED.yaml absent - split digest will not be recorded "
          "(measurement does not need it).")

## 3. Dependencies

In [ ]:
# torch/numpy ship with Colab. These are the modules the trainer actually imports:
# transformers + accelerate (model), scikit-learn (metrics), PyYAML (manifests), pydantic (schema).
%pip install -q "transformers>=4.40" "accelerate>=0.30" "scikit-learn>=1.4" "PyYAML>=6.0" "pydantic>=2.7"

# Fail here, loudly, rather than 30 seconds into training.
import importlib
missing = []
for mod in ("torch", "transformers", "accelerate", "sklearn", "yaml", "pydantic", "numpy"):
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    raise RuntimeError("Missing modules: " + ", ".join(missing) + " - pip install them, then re-run.")
print("all required modules import OK")

import torch, transformers, sklearn
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda        :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 4. Pre-flight checks

Verifies the split digest is the one every earlier result was measured against. If it does not
match, the notebook stops rather than producing numbers that cannot be compared.

In [ ]:

# Fail here rather than at training time if no GPU is attached. Exit code 3 from the trainer means
# exactly this, and it is much cheaper to notice now.
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached to this runtime.\n"
        "  torch.version.cuda is " + str(torch.version.cuda) + " (None = CPU-only build)\n"
        "Runtime -> Change runtime type -> Hardware accelerator: GPU -> Save.\n"
        "That restarts the kernel, so afterwards re-run cells 2, 4, 5 and 7, then this cell.\n"
        "If GPU is selected but you still land on CPU, your Colab GPU quota may be exhausted,\n"
        "or no accelerator was free - check the purchase panel and retry later."
    )
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}  {props.total_memory/1e9:.1f} GB  sm_{props.major}{props.minor}")
import yaml
from pathlib import Path

EXPECTED_DIGEST = "932d20f1dca64c21537bed89739c633ace1b627c1a25f7d78c45a17915cf27a2"

combined_path = REPO / "datasets/manifests/COMBINED.yaml"
if not combined_path.is_file():
    raise RuntimeError("COMBINED.yaml missing - the ZIP must include datasets/manifests/.")

combined = yaml.safe_load(combined_path.read_text(encoding="utf-8"))
digest = combined.get("split_digest")
counts = combined.get("counts", {}).get("records", {})

print("digest :", digest)
print("counts :", counts)
assert digest == EXPECTED_DIGEST, (
    "Split digest changed. Do not train: results would not be comparable to the TF-IDF baselines."
)
print("\nOK - digest matches the baseline run.")

n_train = counts.get("train", 0)
print(f"\nplanned: {n_train} rows x 3 epochs; expect 35-60 min on a T4 "
      f"(measured anchor: 83.6 ex/s at L=128 on this account)")

# Guard against a stale ZIP: the trainer gained --checkpoint-dir/--out-log/--num-workers after the
# measurement ZIP was built. Without them the training cell fails with an argparse error.
trainer_path = REPO / "ml/train/train_encoder.py"
evaluator_path = REPO / "ml/train/evaluate_encoder.py"

if not trainer_path.is_file() or not evaluator_path.is_file():
    raise RuntimeError(
        "The ZIP is stale: ml/train/evaluate_encoder.py or train_encoder.py is missing.\n"
        "Rebuild it with  python scripts/make_colab_zip.py --out StopTheCap.zip  and re-upload."
    )

trainer_src = trainer_path.read_text(encoding="utf-8")
missing = [f for f in ("--checkpoint-dir", "--out-log", "--num-workers") if f not in trainer_src]
if missing:
    raise RuntimeError(
        "The ZIP is stale: train_encoder.py lacks " + ", ".join(missing) + ".\n"
        "Rebuild it with  python scripts/make_colab_zip.py --out StopTheCap.zip  and re-upload to Drive."
    )
print("trainer + evaluator present and current.")

# Data check: without canonical records there is nothing to train on. This is the failure that
# produced "train split is single-class" on the first attempt.
from collections import Counter
from ml.baselines.tfidf import load_text_records
from ml.ingest.schema import Split

records = load_text_records()
if not records:
    raise RuntimeError(
        "datasets/canonical is empty, so the ZIP was built without data."
        " Rebuild it with:  python scripts/make_colab_zip.py --with-data --out StopTheCap.zip"
        "  then re-upload to Drive."
    )

_train = [r for r in records if r.split is Split.TRAIN]
_counts = Counter(r.label for r in _train)
print(f"records loaded: {len(records)}")
print(f"train split   : human={_counts.get(0, 0)}  machine={_counts.get(1, 0)}")
if len(_counts) < 2:
    raise RuntimeError(
        "The train split has only one class. Data is incomplete - rebuild the dataset locally"
        " (python -m ml.cli build) and repackage with --with-data."
    )


## 5. Train E1

Class-weighted loss (human 3.448 / machine 0.585), fp16, batch 16 with grad-accum 2, 3 epochs,
512 tokens. Checkpoints every 200 steps to Drive; re-running resumes from `last.pt`.

In [ ]:
import subprocess, sys
from pathlib import Path

Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
ckpt = Path(RUN_DIR) / "ckpt"
log  = Path(RUN_DIR) / "E1.json"

cmd = [sys.executable, "-m", "ml.train.train_encoder",
       "--device", "cuda",
       "--checkpoint-dir", str(ckpt),
       "--out-log", str(log),
       "--num-workers", "2"]
print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=str(REPO))
print("\nexit code:", proc.returncode)
if proc.returncode != 0:
    print("Training failed - paste the last 30 lines above.")

## 6. Evaluate on all four splits

Threshold is re-derived from `val` at the same 5% human-FPR budget and frozen before any test split
is scored. HC3 and RAID are scored only, never fitted.

In [ ]:
import subprocess, sys
from pathlib import Path

ckpt = Path(RUN_DIR) / "ckpt" / "last.pt"
assert ckpt.is_file(), f"no checkpoint at {ckpt} - did training finish?"

cmd = [sys.executable, "-m", "ml.train.evaluate_encoder",
       "--checkpoint", str(ckpt),
       "--out", "E1_eval"]
print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=str(REPO))
print("\nexit code:", proc.returncode)

## 7. Save results to Drive

In [ ]:
import shutil
from pathlib import Path

run = Path(RUN_DIR)
run.mkdir(parents=True, exist_ok=True)
saved = []
for name in ("E1_eval.json", "E1_eval.md"):
    src = REPO / "reports" / name
    if src.is_file():
        shutil.copy2(src, run / name)
        saved.append(run / name)
print("saved:")
for p in saved:
    print("  ", p, p.stat().st_size, "bytes")
print("\nDownloadable from Drive > StopTheCap_runs > E1.")

## 8. Compare against the TF-IDF baselines

In [ ]:
import json
from pathlib import Path

base_path = REPO / "reports/baseline_tfidf.json"
eval_path = Path(RUN_DIR) / "E1_eval.json"

def fmt(x, nd=3):
    return "n/a" if x is None else f"{x:.{nd}f}"

rows = {}
if base_path.is_file():
    for cfg in json.loads(base_path.read_text()):
        for split, m in cfg.get("metrics", {}).items():
            rows.setdefault(split, {})[f"tfidf-{cfg['config']}"] = m
        for split, m in cfg.get("per_source_quarantine", {}).items():
            rows.setdefault(split, {})[f"tfidf-{cfg['config']}"] = m

if eval_path.is_file():
    for s in json.loads(eval_path.read_text())["splits"]:
        rows.setdefault(s["name"], {})["E1-encoder"] = s

print(f"{'split':18}{'model':16}{'ROC-AUC':>9}{'human FPR':>11}")
for split in sorted(rows):
    for model in sorted(rows[split]):
        m = rows[split][model]
        print(f"{split:18}{model:16}{fmt(m.get('roc_auc')):>9}{fmt(m.get('human_fpr')):>11}")

## 9. How to read the result

- **`test_id` is in-domain.** The TF-IDF baseline scored 0.996 there; the encoder should be
  comparable or better. That number is *not* evidence of general detection ability.
- **`hc3` and `raid` are the honest numbers.** TF-IDF scored 0.652 and 0.480. If the encoder also
  collapses on RAID, that is the finding - report it rather than hiding it.
- **Human FPR is the product metric.** A false accusation is worse than a miss.
- If a split has fewer than 1,000 negatives the evaluator marks its threshold metrics "indicative".

## If it fails

- **OOM:** lower `batch_size` in `ml/train/encoder_config.yaml` to 8 (keep grad_accum 2).
- **Disconnected:** re-run cells 2-5; training resumes from the Drive checkpoint.
- **Digest mismatch:** stop. The manifest changed and numbers are not comparable.